In [ ]:
# pip install llama-index-llms-openai llama-index-vector-stores-chroma


In [4]:

pip install llama-index-embeddings-openai 

  Using cached llama_index_embeddings_openai-0.6.0-py3-none-any.whl.metadata (401 bytes)
Using cached llama_index_embeddings_openai-0.6.0-py3-none-any.whl (7.7 kB)

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import chromadb
from dotenv import load_dotenv

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Settings,
    StorageContext,
)
from llama_index.core.node_parser import SentenceSplitter
# from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.chroma import ChromaVectorStore


In [7]:
load_dotenv()
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0)



In [8]:
docs = SimpleDirectoryReader(
    input_files=[
        "data_test/hr_policy_manual.pdf",
        "data_test/novacrm_product_manual.pdf",
        "data_test/annual_financial_report_2024.pdf",
    ]
).load_data()

In [9]:
nodes = SentenceSplitter(chunk_size=512, chunk_overlap=64).get_nodes_from_documents(docs)


In [18]:
# chroma_client = chromadb.PersistentClient(path="./nb03_chroma")


In [19]:
# col = chroma_client.create_collection("docs_v1")
# vs = ChromaVectorStore(chroma_collection=col)

In [ ]:
# sc = StorageContext.from_defaults(vector_store=vs)




index = VectorStoreIndex(nodes)

2026-06-06 14:02:07,960 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [13]:
query_engine = index.as_query_engine()  

response = query_engine.query("What is the HR Policy Manual?")
print(response)

2026-06-06 14:02:45,576 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-06 14:02:46,856 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The HR Policy Manual is a document that outlines the employment terms, leave policies, and other guidelines for employees at NovaTech Solutions Pvt. Ltd. It includes information on probation periods, notice periods, working hours, annual leave, sick leave, and maternity leave policies.


In [15]:
retriever_basic = index.as_retriever(similarity_top_k=3)

response = retriever_basic.retrieve("What is the HR Policy Manual?")
print(response)

2026-06-06 14:04:31,666 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[NodeWithScore(node=TextNode(id_='ab4a953e-49b1-4751-90ab-d0a406b420fd', embedding=None, metadata={'page_label': '4', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-01', 'last_modified_date': '2026-06-06'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='8ee787e0-4130-4bce-9f19-a69f411ef152', node_type='4', metadata={'page_label': '4', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-01', 'last_modified_date': '2026-06-06'}, hash='9ea0402d21602a54658227aeb17cee36b4d4cb6f8255984302a5a94

In [16]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine(
    retriever=retriever_basic

)

In [17]:
query = "what is the HR Policy Manual?"

response = query_engine.query(query)
print(response)

2026-06-06 14:06:07,660 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-06 14:06:08,803 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The HR Policy Manual is a document that outlines the various policies and guidelines related to employment terms, leave policies, compensation and benefits, performance appraisal cycles, code of conduct, and disciplinary actions at NovaTech Solutions Pvt. Ltd.
